In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q timm split-folders scikit-learn

In [ ]:
import splitfolders

input_dir = "/content/drive/MyDrive/Kits -DS DA internship_2 Aug - Sep /dataset2_for_early stress_images"
output_dir = "/content/drive/MyDrive/split_dataset"

splitfolders.ratio(
    input_dir,
    output=output_dir,
    seed=42,
    ratio=(0.7, 0.2, 0.1)
)

Copying files: 14354 files [08:09, 29.31 files/s]


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import timm
from collections import Counter
from tqdm import tqdm

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 60
LR = 3e-4
NUM_CLASSES = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_DIR = "/content/drive/MyDrive/DeiT_v2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
train_dir = "/content/drive/MyDrive/split_dataset/train"
val_dir   = "/content/drive/MyDrive/split_dataset/val"
test_dir  = "/content/drive/MyDrive/split_dataset/test"

train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir, transform=val_test_transform)
test_ds  = datasets.ImageFolder(test_dir, transform=val_test_transform)

print("Classes:", train_ds.classes)

Classes: ['Healthy', 'boron', 'kalium', 'mg', 'nitrogen']


In [ ]:
class_counts = Counter(train_ds.targets)
num_samples = len(train_ds)

weights = [1.0 / class_counts[label] for label in train_ds.targets]
sampler = WeightedRandomSampler(weights, num_samples, replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2
)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = timm.create_model(
    "deit_small_patch16_224",
    pretrained=True,
    num_classes=NUM_CLASSES
).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

In [ ]:
def load_checkpoint():
    ckpts = sorted(os.listdir(CHECKPOINT_DIR))
    if ckpts:
        path = os.path.join(CHECKPOINT_DIR, ckpts[-1])
        print("Resuming from:", path)
        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        return checkpoint["epoch"] + 1
    return 0

In [ ]:
start_epoch = load_checkpoint()

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    scheduler.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {total_loss:.4f} "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")

    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict()
    }, f"{CHECKPOINT_DIR}/deit_epoch_{epoch}.pth")

100%|██████████| 628/628 [02:49<00:00,  3.71it/s]


Epoch [1/60] Loss: 440.4456 Train Acc: 0.8570 Val Acc: 0.7452


100%|██████████| 628/628 [02:45<00:00,  3.79it/s]


Epoch [2/60] Loss: 408.3036 Train Acc: 0.8791 Val Acc: 0.7365


100%|██████████| 628/628 [02:51<00:00,  3.67it/s]


...

Epoch [58/60] Loss: 353.3606 Train Acc: 0.9175 Val Acc: 0.7320


100%|██████████| 628/628 [02:40<00:00,  3.91it/s]


Epoch [59/60] Loss: 354.9703 Train Acc: 0.9142 Val Acc: 0.7327


100%|██████████| 628/628 [02:41<00:00,  3.89it/s]


Epoch [60/60] Loss: 350.8696 Train Acc: 0.9176 Val Acc: 0.7330


In [14]:
import timm

model = timm.create_model(
    "deit_small_patch16_224",
    pretrained=False,
    num_classes=NUM_CLASSES
).to(DEVICE)

In [13]:
checkpoint = torch.load("/content/drive/MyDrive/DeiT_v2_checkpoints/deit_epoch_20.pth",
                        map_location=DEVICE)

print(checkpoint.keys())

dict_keys(['epoch', 'model', 'optimizer', 'scheduler'])


In [8]:
NUM_CLASSES = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [15]:
model.load_state_dict(checkpoint["model"])
model.eval()

print("✅ Epoch 20 best model loaded successfully")

✅ Epoch 20 best model loaded successfully


In [17]:
TEST_DIR = "/content/drive/MyDrive/split_dataset/test"

In [18]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [19]:
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

class_names = test_dataset.classes
print("Classes:", class_names)

Classes: ['Healthy', 'boron', 'kalium', 'mg', 'nitrogen']


In [20]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n Classification Report (Epoch 20 Best Model)\n")
print(classification_report(all_labels, all_preds, target_names=class_names))


📊 Classification Report (Epoch 20 Best Model)

              precision    recall  f1-score   support

     Healthy       0.54      1.00      0.70        98
       boron       1.00      0.99      0.99       252
      kalium       0.99      0.51      0.67       745
          mg       0.54      0.98      0.69       233
    nitrogen       0.56      1.00      0.72       112

    accuracy                           0.74      1440
   macro avg       0.73      0.90      0.76      1440
weighted avg       0.85      0.74      0.74      1440

